# ex021_Brine_Feedback2D

In [ ]:
%config InlineBackend.figure_format = 'svg'
import sys
from pathlib import Path

CASE_DIR = Path.cwd()
sys.path.insert(0, str(CASE_DIR.parents[1]))
OUTPUT_DIR = CASE_DIR / "output"
SIMULATION_DIR = CASE_DIR / "simulation"
import json
from dataclasses import fields

import flopy
import numpy as np

from examples.ex021_Brine_Feedback2D.run import (
    CHEMISTRY,
    INERT_FRACTION,
    POROSITY,
    SCENARIOS,
    ChemistryConfig,
    Config,
)


def load(label: str, *, chemistry: ChemistryConfig) -> dict:
    p = OUTPUT_DIR / label
    meta = json.loads((p / "metadata.json").read_text())
    if "chemistry" in meta:
        chemistry = ChemistryConfig(**meta["chemistry"])
    config = Config(
        **{k: v for k, v in meta["config"].items() if k in {f.name for f in fields(Config)}}
    )
    r = np.load(p / "results.npy")
    headings = (p / "results_headings.txt").read_text(encoding="utf-8-sig").splitlines()
    times = np.load(p / "results_times.npy")
    phi = np.load(p / "results_porosity.npy")
    k = np.load(p / "results_K.npy")
    aq = np.load(p / "aqueous_transport.npy")
    k0 = np.load(p / "initial_K.npy")
    mineral = np.stack([r[:, headings.index(m)] for m in chemistry.mineral_molar_volumes], axis=1)
    carn = mineral[:, 1]
    depletion = (carn[0] - carn) / carn[0]
    bud = flopy.utils.CellBudgetFile(str(SIMULATION_DIR / label / "flow.bud"), precision="double")
    qx = []
    qz = []
    for t in times[1:]:
        sp = bud.get_data(text="DATA-SPDIS", totim=float(t))[0]
        qx.append(sp["qx"].reshape(config.nz, config.nx))
        qz.append(sp["qz"].reshape(config.nz, config.nx))
    bud.close()
    qx = np.stack([qx[0], *qx])
    qz = np.stack([qz[0], *qz])
    hist = json.loads((p / "history.json").read_text())
    heads = np.load(p / "heads.npy").reshape(-1, config.nz, config.nx)
    heads = np.concatenate([heads[:1], heads])
    krow = meta["components"].index("K")
    potassium = (aq[:, krow] * phi).sum(axis=1) + (
        mineral[:, 1] + mineral[:, 2] + 2 * mineral[:, 4]
    ).sum(axis=1)
    potassium *= config.cell_volume * 1000
    recovered = (
        potassium[0]
        + config.injection_rate * meta["injection_mol_L"][krow] * times * 1000
        - potassium
    )
    return dict(
        meta=meta,
        config=config,
        chemistry=chemistry,
        results=r,
        headings=headings,
        times=times,
        phi=phi,
        k=k,
        k0=k0,
        aq=aq,
        mineral=mineral,
        depletion=depletion,
        qx=qx,
        qz=qz,
        hist=hist,
        heads=heads,
        recovered_K_mol=recovered,
    )


def frame(d, day):
    return np.flatnonzero(np.isclose(d["times"], day, rtol=0.0, atol=1e-07)).item()


def focus(d):
    ix = int(np.argmin(abs(d["config"].x - 300)))
    profiles = np.maximum(d["qx"][:, :, ix], 0.0)
    n = int(np.ceil(d["config"].nz * 0.2))
    return np.sort(profiles, axis=1)[:, -n:].sum(axis=1) / profiles.sum(axis=1)


ds = {s: load(s + "_shallow", chemistry=CHEMISTRY) for s in SCENARIOS}
import string

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm, Normalize, PowerNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.transforms import ScaledTranslation

W = 183 / 25.4
COLORS = ["#7A7F87", "#3E79A8", "#D18A3F", "#7B5EA7"]
LABELS = [
    "S00 · Fixed properties",
    "S10 · Density feedback",
    "S01 · Porosity–K feedback",
    "S11 · Both feedbacks",
]
LINESTYLES = ["--", "-.", ":", "-"]
KMAP = LinearSegmentedColormap.from_list(
    "hydraulic", ["#F6F1E6", "#D8E6DF", "#A8D3CE", "#68B2B7", "#367F9D", "#204E70"]
)
QMAP = LinearSegmentedColormap.from_list(
    "flow", ["#fffdf7", "#cae5dc", "#76bdbb", "#348d9b", "#225f7b"]
)
CMAP = LinearSegmentedColormap.from_list(
    "potassium", ["#fcf9f1", "#f3dfb4", "#e3b97f", "#cf855f", "#a85858"]
)
RMAP = LinearSegmentedColormap.from_list(
    "enhancement", ["#fcfaf5", "#f4e3bc", "#e8bc76", "#cf874b", "#a75c3e"]
)
plt.rcdefaults()
plt.rcParams.update(
    {
        "font.family": ["Arial", "DejaVu Sans"],
        "font.size": 8.5,
        "axes.titlesize": 8.5,
        "axes.labelsize": 8,
        "xtick.labelsize": 7.5,
        "ytick.labelsize": 7.5,
        "legend.fontsize": 7.5,
        "axes.linewidth": 0.6,
        "xtick.major.size": 2.5,
        "ytick.major.size": 2.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "svg.fonttype": "none",
        "savefig.facecolor": "white",
    }
)
from examples.ex021_Brine_Feedback2D.run import MINERAL_VOLUME_FRACTIONS

D = {
    "K0": ds["S00"]["k0"].reshape(ds["S00"]["config"].nz, ds["S00"]["config"].nx),
    "mineral_names": np.array(["Pore space", *MINERAL_VOLUME_FRACTIONS, "Inert solid"]),
    "bulk_volume_percent": np.array([POROSITY, *MINERAL_VOLUME_FRACTIONS.values(), INERT_FRACTION])
    * 100,
}


def _manuscript_layout_10(fig):
    fig.canvas.draw()
    width_pt, height_pt = fig.get_size_inches() * 72
    offsets = [(0, 0), (26.592977, 0), (-17.682054, 0), (0, 0)]
    for ax, (dx, dy) in zip(fig.axes, offsets, strict=True):
        pos = ax.get_position()
        ax.set_position([pos.x0 + dx / width_pt, pos.y0 - dy / height_pt, pos.width, pos.height])


def panel(ax, letter, title):
    ax.set_title(title, loc="left", pad=8, fontweight="normal")
    ax.text(
        0,
        1,
        letter,
        transform=ax.transAxes + ScaledTranslation(-15 / 72, 7 / 72, ax.figure.dpi_scale_trans),
        fontsize=10,
        fontweight="bold",
        va="bottom",
    )


def field(ax, a, cmap, norm, xlabel=True, ylabel=True):
    im = ax.imshow(
        a,
        origin="upper",
        extent=(0, 400, 100, 0),
        aspect="auto",
        interpolation="nearest",
        cmap=cmap,
        norm=norm,
        rasterized=True,
    )
    ax.set(xlim=(0, 400), ylim=(100, 0), xticks=[0, 100, 200, 300, 400], yticks=[0, 50, 100])
    ax.tick_params(labelbottom=xlabel, labelleft=ylabel)
    if xlabel:
        ax.set_xlabel("Distance (m)", labelpad=2)
    if ylabel:
        ax.set_ylabel("Depth (m)", labelpad=2)
    ax.plot([0, 0], [0, 10], color="#3E79A8", lw=3, clip_on=False, solid_capstyle="butt")
    ax.plot([400, 400], [0, 20], color="#C66A45", lw=3, clip_on=False, solid_capstyle="butt")
    return im


def bar(fig, im, rect, label, ticks=None):
    ca = fig.add_axes(rect)
    cb = fig.colorbar(im, cax=ca, orientation="horizontal", ticks=ticks)
    cb.set_label(label, labelpad=3)
    cb.outline.set_visible(False)
    cb.ax.minorticks_off()
    if ticks is not None:
        cb.ax.set_xticklabels([f"{t:g}" for t in ticks])


def finish(fig, n, axes):
    _manuscript_layout_10(fig)
    plt.show()
    plt.close(fig)


def figure10(D):
    fig = plt.figure(figsize=(W, 4.8))
    gs = fig.add_gridspec(
        2,
        2,
        left=0.1,
        right=0.96,
        bottom=0.12,
        top=0.91,
        height_ratios=[1, 1.05],
        hspace=0.85,
        wspace=0.8,
    )
    a = fig.add_subplot(gs[0, :])
    b = fig.add_subplot(gs[1, 0])
    config = fig.add_subplot(gs[1, 1])
    im = field(a, D["K0"], KMAP, LogNorm(0.008, 90))
    panel(a, "a", "")
    a.text(
        0.01,
        1.03,
        "Recharge boundary 0–10 m",
        transform=a.transAxes,
        color="#3E79A8",
        fontsize=7.5,
        va="bottom",
    )
    a.text(
        0.99,
        1.03,
        "Pumping well 0–20 m · 5 m³/d",
        transform=a.transAxes,
        color="#C66A45",
        fontsize=7.5,
        ha="right",
        va="bottom",
    )
    a.set_title("")
    bar(
        fig,
        im,
        [0.3, 0.54, 0.48, 0.021],
        "Initial horizontal hydraulic conductivity (m/d)",
        [0.01, 0.1, 1, 10, 90],
    )
    names = D["mineral_names"]
    vals = D["bulk_volume_percent"]
    b.barh(
        range(7),
        vals,
        height=0.53,
        color=["#80B7B2", "#C7CBCB", "#D49A57", "#D8D2C8", "#B9C6C9", "#C9C1B5", "#8FA5AA"],
    )
    b.set_yticks(range(7), names)
    b.invert_yaxis()
    b.set_xlim(0, 46)
    b.set_xticks([0, 20, 40])
    b.set_xlabel("Bulk volume (%)")
    for i, v in enumerate(vals):
        b.text(v + 0.9, i, f"{v:g}", va="center", fontsize=7.5)
    panel(b, "b", "Finite reactive mineral inventory")
    for i, (x, y) in enumerate([(0, 0), (1, 0), (0, 1), (1, 1)]):
        config.add_patch(
            Rectangle((x, y), 1, 1, facecolor=COLORS[i], alpha=0.16, edgecolor="white", lw=3)
        )
        config.text(
            x + 0.5,
            y + 0.51,
            ["S00", "S10", "S01", "S11"][i],
            ha="center",
            va="center",
            color=COLORS[i],
            fontweight="bold",
            fontsize=11,
        )
    config.set(xlim=(0, 2), ylim=(0, 2), xticks=[0.5, 1.5], yticks=[0.5, 1.5])
    config.set_xticklabels(["Off", "On"])
    config.set_yticklabels(["Off", "On"])
    config.set_xlabel("Density feedback")
    config.set_ylabel("Porosity–K feedback")
    config.tick_params(length=0)
    for sp in config.spines.values():
        sp.set_visible(False)
    panel(config, "c", "Same chemistry, different feedbacks")
    finish(fig, 10, [a, b, config])


figure10(D)

In [ ]:
%config InlineBackend.figure_format = 'svg'
from matplotlib.colors import LinearSegmentedColormap

W = 183 / 25.4
COLORS = ["#80868c", "#427fa3", "#c89348", "#178d89"]
LABELS = [
    "S00 · Fixed properties",
    "S10 · Density feedback",
    "S01 · Porosity–K feedback",
    "S11 · Both feedbacks",
]
LINESTYLES = ["--", "-.", ":", "-"]
KMAP = LinearSegmentedColormap.from_list(
    "hydraulic", ["#fbf5df", "#d9e4cd", "#8cc5b9", "#408f9c", "#256178"]
)
QMAP = LinearSegmentedColormap.from_list(
    "flow", ["#fffdf7", "#cae5dc", "#76bdbb", "#348d9b", "#225f7b"]
)
CMAP = LinearSegmentedColormap.from_list(
    "potassium", ["#fcf9f1", "#f3dfb4", "#e3b97f", "#cf855f", "#a85858"]
)
RMAP = LinearSegmentedColormap.from_list(
    "enhancement", ["#fcfaf5", "#f4e3bc", "#e8bc76", "#cf874b", "#a75c3e"]
)
plt.rcdefaults()
plt.rcParams.update(
    {
        "font.family": ["Arial", "DejaVu Sans"],
        "font.size": 8.5,
        "axes.titlesize": 8.5,
        "axes.labelsize": 8,
        "xtick.labelsize": 7.5,
        "ytick.labelsize": 7.5,
        "legend.fontsize": 7.5,
        "axes.linewidth": 0.6,
        "xtick.major.size": 2.5,
        "ytick.major.size": 2.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "svg.fonttype": "none",
        "savefig.facecolor": "white",
    }
)
scenarios = list(SCENARIOS)
config = ds["S00"]["config"]
D = {
    "K0": ds["S00"]["k0"].reshape(config.nz, config.nx),
    "x": config.x,
    "depth": config.height - config.z,
}
for key in ["depletion", "solute_K", "heads", "qx", "qz"]:
    frames = []
    for s in scenarios:
        d = ds[s]
        idx = frame(d, 1600)
        value = (
            d["aq"][idx, d["meta"]["components"].index("K")] if key == "solute_K" else d[key][idx]
        )
        frames.append(value.reshape(config.nz, config.nx))
    D[key] = np.stack(frames)


def _manuscript_layout_11(fig):
    fig.canvas.draw()
    width_pt, height_pt = fig.get_size_inches() * 72
    offsets = [
        (0, 0),
        (0, 0),
        (0, -21.04069),
        (0, -21.04069),
        (0, -41.959318),
        (0, -41.959318),
        (0, -63),
        (0, -63),
        (0, -69),
        (0, -69),
    ]
    for ax, (dx, dy) in zip(fig.axes, offsets, strict=True):
        pos = ax.get_position()
        ax.set_position([pos.x0 + dx / width_pt, pos.y0 - dy / height_pt, pos.width, pos.height])
    for legend in fig.legends:
        box = legend.get_bbox_to_anchor().transformed(fig.transFigure.inverted())
        legend.set_bbox_to_anchor(
            (box.x0 + 0 / width_pt, box.y0 - -69 / height_pt, box.width, box.height),
            transform=fig.transFigure,
        )


def panel_section9(ax, letter, title):
    ax.set_title(title, loc="left", pad=8, fontweight="normal")
    ax.text(
        0,
        1,
        letter,
        transform=ax.transAxes + ScaledTranslation(-15 / 72, 7 / 72, ax.figure.dpi_scale_trans),
        fontsize=10,
        fontweight="bold",
        va="bottom",
    )


def field_section9(ax, a, cmap, norm, xlabel=True, ylabel=True):
    im = ax.imshow(
        a,
        origin="upper",
        extent=(0, 400, 100, 0),
        aspect="auto",
        interpolation="nearest",
        cmap=cmap,
        norm=norm,
        rasterized=True,
    )
    ax.set(xlim=(0, 400), ylim=(100, 0), xticks=[0, 100, 200, 300, 400], yticks=[0, 50, 100])
    ax.tick_params(labelbottom=xlabel, labelleft=ylabel)
    if xlabel:
        ax.set_xlabel("Distance (m)", labelpad=2)
    if ylabel:
        ax.set_ylabel("Depth (m)", labelpad=2)
    ax.plot([0, 0], [0, 10], color="#427fa3", lw=3, clip_on=False, solid_capstyle="butt")
    ax.plot([400, 400], [0, 20], color="#b8674b", lw=3, clip_on=False, solid_capstyle="butt")
    return im


def bar_section9(fig, im, rect, label, ticks=None):
    ca = fig.add_axes(rect)
    cb = fig.colorbar(im, cax=ca, orientation="horizontal", ticks=ticks)
    cb.set_label(label, labelpad=3)
    cb.outline.set_visible(False)
    cb.ax.minorticks_off()
    if ticks is not None:
        cb.ax.set_xticklabels([f"{t:g}" for t in ticks])


def finish_section9(fig, n, axes):
    _manuscript_layout_11(fig)
    plt.show()
    plt.close(fig)


def figure11(D):
    fig, axes = plt.subplots(4, 2, figsize=(W, 7.05))
    fig.subplots_adjust(left=0.095, right=0.98, bottom=0.18, top=0.94, wspace=0.22, hspace=0.7)
    cmin = float(D["solute_K"].min())
    cmax = float(D["solute_K"].max())
    hlevels = [115, 125, 135, 145]
    hstyles = [":", "--", "-.", "-"]
    for row in range(4):
        a, b = axes[row]
        speed = np.hypot(D["qx"][row], D["qz"][row])
        im1 = field_section9(a, speed, QMAP, PowerNorm(gamma=0.5, vmin=0, vmax=1), xlabel=row == 3)
        a.contour(
            D["x"], D["depth"], D["depletion"][row], levels=[0.5], colors="#b8793e", linewidths=0.8
        )
        im2 = field_section9(
            b, D["solute_K"][row], CMAP, Normalize(cmin, cmax), xlabel=row == 3, ylabel=False
        )
        b.contour(
            D["x"],
            D["depth"],
            D["heads"][row],
            levels=hlevels,
            colors="#59676d",
            linewidths=0.6,
            linestyles=hstyles,
            alpha=0.8,
        )
        panel_section9(a, string.ascii_lowercase[2 * row], LABELS[row])
        panel_section9(b, string.ascii_lowercase[2 * row + 1], "Dissolved K and hydraulic head")
    bar_section9(
        fig,
        im1,
        [0.135, 0.095, 0.32, 0.018],
        "Darcy flux magnitude (m/d)",
        [0, 0.01, 0.05, 0.2, 0.5, 1],
    )
    bar_section9(fig, im2, [0.61, 0.095, 0.32, 0.018], "Dissolved K (mol/L)")
    handles = [Line2D([], [], color="#b8793e", lw=0.8, label="50% removal")] + [
        Line2D([], [], color="#59676d", ls=ls, lw=0.8, label=f"{h} m head")
        for h, ls in zip(hlevels, hstyles, strict=False)
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=5,
        bbox_to_anchor=(0.53, 0.005),
        columnspacing=1.2,
        handlelength=1.8,
    )
    finish_section9(fig, 11, list(axes.flat))


figure11(D)

In [ ]:
%config InlineBackend.figure_format = 'svg'
W = 183 / 25.4
COLORS = ["#80868c", "#427fa3", "#c89348", "#178d89"]
LABELS = [
    "S00 · Fixed properties",
    "S10 · Density feedback",
    "S01 · Porosity–K feedback",
    "S11 · Both feedbacks",
]
LINESTYLES = ["--", "-.", ":", "-"]
KMAP = LinearSegmentedColormap.from_list(
    "hydraulic", ["#fbf5df", "#d9e4cd", "#8cc5b9", "#408f9c", "#256178"]
)
QMAP = LinearSegmentedColormap.from_list(
    "flow", ["#fffdf7", "#cae5dc", "#76bdbb", "#348d9b", "#225f7b"]
)
CMAP = LinearSegmentedColormap.from_list(
    "potassium", ["#fcf9f1", "#f3dfb4", "#e3b97f", "#cf855f", "#a85858"]
)
RMAP = LinearSegmentedColormap.from_list(
    "enhancement", ["#fcfaf5", "#f4e3bc", "#e8bc76", "#cf874b", "#a75c3e"]
)
plt.rcdefaults()
plt.rcParams.update(
    {
        "font.family": ["Arial", "DejaVu Sans"],
        "font.size": 8.5,
        "axes.titlesize": 8.5,
        "axes.labelsize": 8,
        "xtick.labelsize": 7.5,
        "ytick.labelsize": 7.5,
        "legend.fontsize": 7.5,
        "axes.linewidth": 0.6,
        "xtick.major.size": 2.5,
        "ytick.major.size": 2.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "svg.fonttype": "none",
        "savefig.facecolor": "white",
    }
)
config = ds["S11"]["config"]
d = ds["S11"]
evolution_days = np.array([400, 2400])
idx = [frame(d, t) for t in evolution_days]
D = {
    "x": config.x,
    "depth": config.height - config.z,
    "time": d["times"],
    "evolution_days": evolution_days,
    "ratio": (d["k"][idx] / d["k0"]).reshape(-1, config.nz, config.nx),
    "evolution_qx": d["qx"][idx],
    "evolution_qz": d["qz"][idx],
}
D["removed"] = np.stack([d["depletion"].mean(axis=1) for d in ds.values()])
D["recovered"] = np.stack([d["recovered_K_mol"] for d in ds.values()])
D["head_excess"] = np.stack(
    [
        d["heads"][:, d["config"].inlet_layers, 0].mean(axis=1) - d["config"].outlet_head
        for d in ds.values()
    ]
)
D["focus"] = np.stack([focus(d) for d in ds.values()])


def _manuscript_layout_12(fig):
    fig.canvas.draw()
    width_pt, height_pt = fig.get_size_inches() * 72
    offsets = [
        (0, 0),
        (0, 0),
        (-14.807232, -4.5),
        (-14.807232, -4.5),
        (-14.807232, -4.5),
        (-14.807232, -4.5),
        (0, -25.461447),
    ]
    for ax, (dx, dy) in zip(fig.axes, offsets, strict=True):
        pos = ax.get_position()
        ax.set_position([pos.x0 + dx / width_pt, pos.y0 - dy / height_pt, pos.width, pos.height])
    for legend in fig.legends:
        box = legend.get_bbox_to_anchor().transformed(fig.transFigure.inverted())
        legend.set_bbox_to_anchor(
            (box.x0 + -14.807232 / width_pt, box.y0 - -15 / height_pt, box.width, box.height),
            transform=fig.transFigure,
        )


def panel_section11(ax, letter, title):
    ax.set_title(title, loc="left", pad=8, fontweight="normal")
    ax.text(
        0,
        1,
        letter,
        transform=ax.transAxes + ScaledTranslation(-15 / 72, 7 / 72, ax.figure.dpi_scale_trans),
        fontsize=10,
        fontweight="bold",
        va="bottom",
    )


def field_section11(ax, a, cmap, norm, xlabel=True, ylabel=True):
    im = ax.imshow(
        a,
        origin="upper",
        extent=(0, 400, 100, 0),
        aspect="auto",
        interpolation="nearest",
        cmap=cmap,
        norm=norm,
        rasterized=True,
    )
    ax.set(xlim=(0, 400), ylim=(100, 0), xticks=[0, 100, 200, 300, 400], yticks=[0, 50, 100])
    ax.tick_params(labelbottom=xlabel, labelleft=ylabel)
    if xlabel:
        ax.set_xlabel("Distance (m)", labelpad=2)
    if ylabel:
        ax.set_ylabel("Depth (m)", labelpad=2)
    ax.plot([0, 0], [0, 10], color="#427fa3", lw=3, clip_on=False, solid_capstyle="butt")
    ax.plot([400, 400], [0, 20], color="#b8674b", lw=3, clip_on=False, solid_capstyle="butt")
    return im


def streams(ax, x, depth, qx, qz):
    ax.streamplot(
        x,
        depth,
        qx,
        -qz,
        color="#4f656a",
        density=(0.6, 0.4),
        linewidth=0.43,
        arrowsize=0.52,
        broken_streamlines=True,
    )


def bar_section11(fig, im, rect, label, ticks=None):
    ca = fig.add_axes(rect)
    cb = fig.colorbar(im, cax=ca, orientation="horizontal", ticks=ticks)
    cb.set_label(label, labelpad=3)
    cb.outline.set_visible(False)
    cb.ax.minorticks_off()
    if ticks is not None:
        cb.ax.set_xticklabels([f"{t:g}" for t in ticks])


def finish_section11(fig, n, axes):
    _manuscript_layout_12(fig)
    plt.show()
    plt.close(fig)


def figure12(D):
    fig = plt.figure(figsize=(W, 4.85))
    gs = fig.add_gridspec(
        2,
        4,
        left=0.11,
        right=0.97,
        bottom=0.17,
        top=0.94,
        wspace=0.62,
        hspace=0.95,
        height_ratios=[1, 1],
    )
    axes = np.empty(6, dtype=object)
    axes[0] = fig.add_subplot(gs[0, 0:2])
    axes[1] = fig.add_subplot(gs[0, 2:4])
    axes[2] = fig.add_subplot(gs[1, 0])
    axes[3] = fig.add_subplot(gs[1, 1])
    axes[4] = fig.add_subplot(gs[1, 2])
    axes[5] = fig.add_subplot(gs[1, 3])
    for j, day in enumerate(D["evolution_days"]):
        ax = axes[j]
        im = field_section11(
            ax, D["ratio"][j], RMAP, LogNorm(1, max(12.0, float(D["ratio"].max()))), ylabel=j == 0
        )
        streams(ax, D["x"], D["depth"], D["evolution_qx"][j], D["evolution_qz"][j])
        panel_section11(ax, string.ascii_lowercase[j], f"Both feedbacks · {day:,} d")
    bar_section11(
        fig, im, [0.33, 0.505, 0.42, 0.016], "Hydraulic conductivity ratio K/K₀", [1, 2, 4, 8, 12]
    )
    titles = [
        "Mineral mobilization",
        "Potassium produced",
        "Hydraulic response",
        "Preferential flow at x ≈ 300 m",
    ]
    ylabels = [
        "Carnallite removed (%)",
        "Cumulative K (Mmol)",
        "Head difference (m)",
        "Top-20% flux share (%)",
    ]
    series = [D["removed"] * 100, D["recovered"] / 1000000.0, D["head_excess"], D["focus"] * 100]
    for j, ax in enumerate(axes[2:]):
        for i in range(4):
            ax.plot(
                D["time"],
                series[j][i],
                color=COLORS[i],
                ls=LINESTYLES[i],
                lw=1.6 if i == 3 else 1.2,
            )
        ax.set_xlim(0, D["time"][-1])
        ax.set_xticks(np.linspace(0, D["time"][-1], 4))
        ax.set_xlabel("Time (d)", labelpad=3)
        ax.set_ylabel(ylabels[j], labelpad=4)
        ax.grid(axis="y", color="#e6ebeb", lw=0.45)
        ax.set_axisbelow(True)
        panel_section11(ax, string.ascii_lowercase[j + 2], titles[j])
        if j == 3:
            ax.set_ylim(20, 100)
            ax.set_yticks([20, 40, 60, 80, 100])
    handles = [
        Line2D(
            [],
            [],
            color=COLORS[i],
            ls=LINESTYLES[i],
            lw=1.5,
            label=["S00 Fixed", "S10 Density", "S01 Porosity–K", "S11 Both"][i],
        )
        for i in range(4)
    ]
    fig.legend(
        handles=handles, loc="lower center", ncol=4, bbox_to_anchor=(0.53, 0.015), columnspacing=1.5
    )
    finish_section11(fig, 12, list(axes))


figure12(D)